# Chapter 09 — Dimensionality Reduction: Live Examples

**Session 3 | Chapter 3 | Duration: ~10 min**

We apply PCA and t-SNE to the handwritten digits dataset (64 features → 2D visualization).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. The Dataset: Handwritten Digits (64 features)

In [ ]:
digits = load_digits()
X = digits.data          # 64 features (8x8 pixel values)
y = digits.target        # 0-9

print(f'Shape: {X.shape}  →  {X.shape[0]} samples, {X.shape[1]} features')
print('Each sample = an 8×8 grayscale image flattened to 64 numbers')

## 2. PCA: Scree Plot — How Many Components?

In [ ]:
# Scale first (mandatory for PCA)
X_scaled = StandardScaler().fit_transform(X)

# Full PCA (all components)
pca_full = PCA()
pca_full.fit(X_scaled)

cumvar = pca_full.explained_variance_ratio_.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
axes[0].bar(range(1, 21), pca_full.explained_variance_ratio_[:20], color='#3498db', alpha=0.8)
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Variance Explained by Each Component', fontsize=13)
axes[0].set_xticks(range(1, 21))

# Cumulative
axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'o-', color='#e74c3c', linewidth=2)
for thresh in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cumvar >= thresh) + 1
    axes[1].axhline(thresh, color='gray', linestyle='--', alpha=0.6)
    axes[1].annotate(f'{thresh:.0%}: {n_comp} components',
                     xy=(n_comp, thresh), xytext=(n_comp + 2, thresh - 0.04),
                     fontsize=9, color='gray')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Variance — Choose your cutoff', fontsize=13)
axes[1].set_xlim(1, 40)

plt.suptitle('PCA Scree Plot — 64 Features Ranked by Importance', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

for thresh in [0.80, 0.90, 0.95]:
    n_comp = np.argmax(cumvar >= thresh) + 1
    print(f'  {thresh:.0%} of variance → {n_comp} components ({n_comp}/{X.shape[1]} features)')

## 3. PCA: 2D Visualization

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
                     c=y, cmap='tab10', alpha=0.6, s=25)
plt.colorbar(scatter, ax=ax, label='Digit')

# Annotate cluster centers
for digit in range(10):
    mask = y == digit
    cx = X_pca[mask, 0].mean()
    cy = X_pca[mask, 1].mean()
    ax.annotate(str(digit), (cx, cy), fontsize=13, fontweight='bold',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

var_explained = pca_2d.explained_variance_ratio_.sum()
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title(f'PCA: 64-dim Digits projected to 2D ({var_explained:.1%} variance retained)', fontsize=13)
plt.tight_layout()
plt.show()

print('We can already see digit clusters separating in 2D.')
print('0 and 1 are very distinct. 4 and 9 overlap (similar shapes).')

## 4. t-SNE: Non-linear 2D Visualization

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (title, X_2d, method) in zip(axes, [
    ('PCA (linear)', X_pca, 'PC'),
    ('t-SNE (non-linear)', X_tsne, 'Dim')
]):
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab10', alpha=0.7, s=20)
    for digit in range(10):
        mask = y == digit
        cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
        ax.annotate(str(digit), (cx, cy), fontsize=13, fontweight='bold', ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(f'{method} 1')
    ax.set_ylabel(f'{method} 2')

plt.colorbar(scatter, ax=axes[1], label='Digit')
plt.suptitle('PCA vs t-SNE: Visualizing 64-dimensional Digit Data', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('t-SNE creates much more separated clusters!')
print('BUT: the distances between clusters in t-SNE are NOT meaningful.')
print('Only use t-SNE for visualization, not as features for ML.')

## 5. PCA as Preprocessing — Does It Help Classification?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Without PCA
rf_nopca = Pipeline([('s', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
score_nopca = cross_val_score(rf_nopca, X, y, cv=5).mean()

# With PCA (95% variance)
rf_pca = Pipeline([
    ('s', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
score_pca = cross_val_score(rf_pca, X, y, cv=5).mean()

rf_pca_full = rf_pca.fit(X_train, y_train)
n_components_used = rf_pca_full['pca'].n_components_

print(f'Random Forest WITHOUT PCA:  CV accuracy = {score_nopca:.4f}  (64 features)')
print(f'Random Forest WITH PCA 95%: CV accuracy = {score_pca:.4f}  ({n_components_used} components)')
print()
print(f'Used {n_components_used}/{X.shape[1]} features to explain 95% of variance')
print('→ Similar performance with fewer features = faster, simpler model')

## Summary

| Technique | Dimensions In | Dimensions Out | Use For |
|-----------|:---:|:---:|--------|
| PCA (n_components=2) | 64 | 2 | Visualization |
| PCA (n_components=0.95) | 64 | ~20 | Preprocessing |
| t-SNE | 64 | 2 | Visualization only |

- PCA: linear, fast, interpretable, for both visualization and preprocessing
- t-SNE: non-linear, stunning visualizations, NEVER use as ML features